# Atelier Preparation de Donnees Textuelles

## Partie 1 - Exploration du corpus

### 1) Chargement des donnees CSV

In [ ]:
import pandas as pd

df = pd.read_csv("../data/smart_reviews_raw.csv", encoding="utf-8")
df.head()

### 2) et 3) Nombre d'avis et nombre de colonnes

In [ ]:
n_avis, n_colonnes = df.shape
print(f"Nombre d'avis : {n_avis}")
print(f"Nombre de colonnes : {n_colonnes}")

### 4) Type de chaque colonne

In [ ]:
df.dtypes

### 5) Valeurs manquantes

In [ ]:
df.isna().sum()

### 6) Identifier quelques types de texte

In [ ]:
import re

texte = df["texte"]

exemples = {
    "normal": texte[texte.str.len().between(30, 60) & texte.notna()].iloc[0],
    "vide": texte[texte.isna()].index[0] if texte.isna().any() else None,
    "url": texte[texte.str.contains("http", na=False)].iloc[0],
    "mention": texte[texte.str.contains("@", na=False)].iloc[0],
    "hashtag": texte[texte.str.contains("#", na=False)].iloc[0],
    "emoji": texte[texte.str.contains(r"[\U0001F300-\U0001FAFF]", na=False, regex=True)].iloc[0],
    "ponctuation": texte[texte.str.contains(r"[!?]{2,}", na=False, regex=True)].iloc[0],
    "majuscules": texte[texte.str.isupper().fillna(False)].iloc[0],
    "repetition": texte[texte.str.contains(r"(.)\1{2,}", na=False, regex=True)].iloc[0],
}

for type_texte, exemple in exemples.items():
    print(f"{type_texte:12s}: {exemple}")

### 7) Longueur des textes

In [ ]:
df["longueur"] = df["texte"].str.len()

stats_longueur = df["longueur"].describe()
print(f"Longueur minimale : {df['longueur'].min()}")
print(f"Longueur maximale : {df['longueur'].max()}")
print(f"Longueur moyenne  : {df['longueur'].mean():.2f}")
print(f"Longueur mediane  : {df['longueur'].median()}")
print(f"Q1 : {df['longueur'].quantile(0.25)}")
print(f"Q3 : {df['longueur'].quantile(0.75)}")
stats_longueur

### 8) Visualisation de la distribution de la longueur des avis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["longueur"].dropna(), bins=40, color="steelblue")
axes[0].set_title("Distribution de la longueur des avis")
axes[0].set_xlabel("Longueur (caracteres)")
axes[0].set_ylabel("Frequence")

axes[1].boxplot(df["longueur"].dropna(), vert=True)
axes[1].set_title("Boxplot de la longueur des avis")
axes[1].set_ylabel("Longueur (caracteres)")

plt.tight_layout()
plt.show()

**a) Existe-t-il des textes anormalement longs ?**

Oui. La moyenne se situe autour de 49 caracteres et le 3e quartile autour de 56, mais le maximum atteint plus de 600 caracteres. Ces valeurs tres eloignees de la distribution principale (visibles comme points isoles au-dessus de la moustache superieure du boxplot) constituent des outliers a examiner : ils peuvent correspondre a des avis tres detailles, du texte duplique/spam, ou du contenu concatene par erreur lors de la collecte.

**b) Existe-t-il beaucoup de textes tres courts ?**

Oui, une partie non negligeable des avis a une longueur proche du minimum observe (2 caracteres), ce qui correspond a des textes quasi vides ou peu informatifs (ex: un seul mot, une ponctuation). Ces textes tres courts risquent d'apporter peu de signal pour la classification de sentiment et devront etre surveilles lors du nettoyage.

### 9) Detection des textes vides

In [ ]:
textes_vides = df[df["texte"].isna() | (df["texte"].str.strip() == "")]
print(f"Nombre de textes vides : {len(textes_vides)}")
textes_vides[["id_avis", "texte"]]

### 10) Detection des doublons sur le texte

In [ ]:
doublons = df[df.duplicated(subset="texte", keep=False) & df["texte"].notna()]
print(f"Nombre de lignes impliquees dans un doublon de texte : {len(doublons)}")
print(f"Nombre de doublons (hors 1re occurrence) : {df['texte'].duplicated().sum()}")
doublons.sort_values("texte")[["id_avis", "texte", "source"]].head(10)

### 11) et 12) Equilibre des classes et classe majoritaire

In [ ]:
sentiment_counts = df["sentiment"].value_counts()
sentiment_pct = df["sentiment"].value_counts(normalize=True) * 100

print(sentiment_counts)
print()
print(sentiment_pct.round(1))

fig, ax = plt.subplots(figsize=(5, 4))
sentiment_counts.plot(kind="bar", color="steelblue", ax=ax)
ax.set_title("Repartition des classes de sentiment")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Classe majoritaire : {sentiment_counts.idxmax()} ({sentiment_counts.max()} avis)")

Les classes ne sont **pas equilibrees** : la classe `positif` domine largement (~65% des avis), suivie de `neutre` (~20%) puis `negatif` (~15%). Ce dessequilibre doit etre pris en compte lors de l'entrainement d'un modele (risque de biais vers la classe majoritaire) et lors du choix des metriques d'evaluation.

### 13) Metriques de classification a envisager

Le dataset etant desequilibre (classe `positif` largement majoritaire), l'**accuracy seule** serait trompeuse : un modele qui predirait toujours `positif` obtiendrait deja un score eleve sans etre utile.

Metriques a privilegier :
- **F1-score macro** : moyenne non ponderee du F1 par classe, donne le meme poids a chaque classe (positif, neutre, negatif) quelle que soit sa frequence.
- **F1-score pondere (weighted)** : tient compte du poids reel de chaque classe, utile pour une vue d'ensemble realiste.
- **Precision et rappel par classe** : essentiels pour verifier que la classe minoritaire (`negatif`) n'est pas sacrifiee.
- **Matrice de confusion** : permet de visualiser les confusions entre classes (ex: neutre confondu avec positif).

L'accuracy peut etre suivie en complement mais ne doit pas etre la metrique de decision principale.

### 14) Detection des caracteres particuliers

In [ ]:
patterns = {
    "emojis": r"[\U0001F300-\U0001FAFF]",
    "urls": r"https?://\S+|www\.\S+",
    "hashtags": r"#\w+",
    "mentions": r"@\w+",
    "chiffres": r"\d",
    "caracteres_speciaux": r"[^\w\s]",
    "ponctuation_repetee": r"[!?.]{2,}",
}

texte = df["texte"]
resultats = {
    nom: texte.str.contains(regex, na=False, regex=True).sum()
    for nom, regex in patterns.items()
}

for nom, count in resultats.items():
    print(f"{nom:22s}: {count} textes ({count / len(texte) * 100:.1f}%)")

### 15) Nombre d'avis par source

In [ ]:
source_counts = df["source"].value_counts()
print(source_counts)

fig, ax = plt.subplots(figsize=(6, 4))
source_counts.plot(kind="bar", color="darkorange", ax=ax)
ax.set_title("Nombre d'avis par source")
ax.set_xlabel("Source")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Les avis sont globalement bien repartis entre les 5 sources attendues (`web`, `mobile`, `sav`, `email`, `reseaux_sociaux`), chacune representant environ 20% du dataset. On remarque toutefois une **incoherence de casse** : la valeur `WEB` apparait une fois en plus de `web`, ce qui cree artificiellement une 6e categorie. Cette incoherence devra etre harmonisee (mise en minuscules) lors du nettoyage pour ne pas fausser les analyses par source.

### 16) Nombre d'avis par produit

In [ ]:
produit_counts = df["produit"].value_counts()
print(produit_counts)

fig, ax = plt.subplots(figsize=(7, 4))
produit_counts.plot(kind="bar", color="seagreen", ax=ax)
ax.set_title("Nombre d'avis par produit")
ax.set_xlabel("Produit")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Les 6 produits (Ordinateur NovaBook, SmartPhone X, SmartPhone Y, Ecouteurs AirSound, SmartWatch Pro, Tablette TabPlus) recoivent un nombre d'avis relativement homogene, entre ~180 et ~220 avis chacun. Il n'y a pas de produit sous-represente de maniere critique, ce qui permettra une analyse par produit fiable sans necessiter de reequilibrage specifique sur cet axe.

## Partie 2 - Nettoyage

### 1) Suppression des lignes avec texte manquant

In [ ]:
print(f"Nombre de lignes avant : {len(df)}")

df_clean = df.dropna(subset=["texte"]).copy()

print(f"Nombre de lignes apres : {len(df_clean)}")
print(f"Lignes supprimees : {len(df) - len(df_clean)}")

### 2) Suppression des doublons

In [ ]:
n_avant = len(df_clean)
print(f"Nombre de lignes avant : {n_avant}")

df_clean = df_clean.drop_duplicates(subset="texte", keep="first").copy()

n_apres = len(df_clean)
print(f"Nombre de lignes apres : {n_apres}")
print(f"Doublons supprimes : {n_avant - n_apres}")

### 3) Suppression des URLs

In [ ]:
def supprimer_urls(texte):
    return re.sub(r"https?://\S+|www\.\S+", "", texte)

exemple_avant = df_clean[df_clean["texte"].str.contains("http", na=False)]["texte"].iloc[0]
exemple_apres = supprimer_urls(exemple_avant)

print(f"Avant : {exemple_avant}")
print(f"Apres : {exemple_apres}")

### 4) Traitement des mentions et hashtags

**Strategie retenue :**
- **Mentions (`@utilisateur`)** : suppression complete. Un nom d'utilisateur ou de marque mentionne n'apporte aucun signal de sentiment et introduit du bruit (identifiants uniques, peu generalisables).
- **Hashtags (`#mot`)** : on **conserve le mot** en supprimant uniquement le symbole `#`. En effet, un hashtag comme `#decu` ou `#genial` peut porter une charge emotionnelle forte et utile pour la classification de sentiment ; le supprimer entierement ferait perdre de l'information pertinente.

In [ ]:
def traiter_mentions_hashtags(texte):
    texte = re.sub(r"@\w+", "", texte)
    texte = re.sub(r"#(\w+)", r"\1", texte)
    return texte

exemple_mention = df_clean[df_clean["texte"].str.contains("@", na=False)]["texte"].iloc[0]
exemple_hashtag = df_clean[df_clean["texte"].str.contains("#", na=False)]["texte"].iloc[0]

print(f"Avant (mention) : {exemple_mention}")
print(f"Apres (mention) : {traiter_mentions_hashtags(exemple_mention)}")
print()
print(f"Avant (hashtag) : {exemple_hashtag}")
print(f"Apres (hashtag) : {traiter_mentions_hashtags(exemple_hashtag)}")

### 5) Nettoyage des espaces

In [ ]:
def nettoyer_espaces(texte):
    return re.sub(r"\s+", " ", texte).strip()

exemple_espaces = df_clean[df_clean["texte"].str.contains(r"  ", na=False, regex=True)]["texte"].iloc[0]

print(f"Avant : {exemple_espaces!r}")
print(f"Apres : {nettoyer_espaces(exemple_espaces)!r}")

### 6) Traitement de la ponctuation

**Strategie retenue :** on **conserve** les signes `!` et `?` car ils portent souvent un signal de sentiment (enthousiasme, insatisfaction, question du client), mais on **reduit les repetitions** (`!!!`, `...`, `???`) a une seule occurrence. En effet, la repetition excessive gonfle artificiellement le vocabulaire (`!`, `!!`, `!!!` deviendraient des tokens distincts) sans ajouter d'information utile a un modele bag-of-words/TF-IDF ; l'intensite emotionnelle peut etre capturee autrement (ex: feature de longueur ou de comptage). Les autres signes de ponctuation neutres (virgules, points simples) sont conserves tels quels a ce stade.

In [ ]:
def reduire_ponctuation_repetee(texte):
    return re.sub(r"([!?.])\1{1,}", r"\1", texte)

exemple_ponct = df_clean[df_clean["texte"].str.contains(r"[!?]{2,}", na=False, regex=True)]["texte"].iloc[0]

print(f"Avant : {exemple_ponct!r}")
print(f"Apres : {reduire_ponctuation_repetee(exemple_ponct)!r}")

### 7) Fonction de nettoyage globale

In [ ]:
def nettoyer_texte(texte):
    texte = supprimer_urls(texte)
    texte = traiter_mentions_hashtags(texte)
    texte = reduire_ponctuation_repetee(texte)
    texte = nettoyer_espaces(texte)
    return texte

exemple_combine = (
    "  Super produit !!! Voir https://example.com/avis  @service_client #genial   "
)

print(f"Avant : {exemple_combine!r}")
print(f"Apres : {nettoyer_texte(exemple_combine)!r}")

### 8) Application du nettoyage : colonne texte_clean

In [ ]:
df_clean["texte_clean"] = df_clean["texte"].apply(nettoyer_texte)
df_clean[["texte", "texte_clean"]].head()

### 9) Verification texte vs texte_clean (10 premieres observations, fin Partie 2)

In [ ]:
pd.set_option("display.max_colwidth", None)
df_clean[["texte", "texte_clean"]].head(10)

## Partie 3 - Tokenisation

### 1) Pourquoi split() n'est pas suffisant pour une vraie application NLP

`str.split()` se contente de decouper sur les espaces, ce qui pose plusieurs problemes :
- **Ponctuation collee aux mots** : `"parfait!"` reste un seul token au lieu de `["parfait", "!"]`.
- **Contractions et apostrophes** : `"aujourd'hui"` ou `"c'est"` ne sont pas decoupes de maniere linguistiquement correcte.
- **Emojis et caracteres speciaux colles au texte** : pas de separation propre entre un mot et un emoji accole.
- **Pas de gestion de la casse ni de normalisation** : `"Produit"` et `"produit"` restent deux tokens differents.
- **Pas de reconnaissance d'entites** (URLs, hashtags, nombres) : tout est traite comme du texte brut.

Un tokenizer NLP dedie comme celui de **NLTK** applique des regles linguistiques (separation ponctuation/mots, gestion des contractions, etc.) et constitue une base plus fiable avant normalisation et vectorisation.

### 2) Tokenisation avec NLTK (fin Partie 3)

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import word_tokenize

df_clean["tokens"] = df_clean["texte_clean"].apply(lambda t: word_tokenize(t, language="french"))
df_clean["n_tokens"] = df_clean["tokens"].apply(len)

print(f"Nombre moyen de tokens : {df_clean['n_tokens'].mean():.2f}")

idx_max = df_clean["n_tokens"].idxmax()
idx_min = df_clean["n_tokens"].idxmin()

print(f"\nTexte avec le plus de tokens ({df_clean.loc[idx_max, 'n_tokens']}) :")
print(df_clean.loc[idx_max, "texte_clean"])

print(f"\nTexte avec le moins de tokens ({df_clean.loc[idx_min, 'n_tokens']}) :")
print(df_clean.loc[idx_min, "texte_clean"])

df_clean[["texte_clean", "tokens", "n_tokens"]].head()

## Partie 4 - Normalisation

### 1) Uniformisation de la casse en minuscules

In [ ]:
df_clean["texte_norm"] = df_clean["texte_clean"].str.lower()
df_clean[["texte_clean", "texte_norm"]].head()

**Pourquoi ce traitement ?**

Sans uniformisation, des mots identiques mais ecrits avec une casse differente (`"Produit"`, `"produit"`, `"PRODUIT"`) seraient consideres comme des tokens distincts par un vectoriseur (Bag of Words, TF-IDF). Cela gonfle artificiellement le vocabulaire, dilue les frequences des mots et degrade la qualite des representations numeriques. Passer en minuscules reduit la dimensionnalite et regroupe les occurrences d'un meme mot sous un seul token, ameliorant la generalisation du modele.

### 2) Faut-il systematiquement supprimer les accents dans un corpus francais ?

**Non.** En francais, les accents portent du sens et distinguent des mots differents : `"sale"` vs `"sale"` (sans accent) sont deux mots distincts, de meme que `"eleve"` (adjectif/nom) vs des formes accentuees comme `"elevee"`. Supprimer systematiquement les accents peut donc **fusionner par erreur des mots de sens different**, creer des ambiguites et degrader la qualite semantique du corpus.

La suppression des accents ne se justifie que dans des cas particuliers : corpus tres bruite avec de nombreuses fautes d'accentuation, texte issu de sources ou les accents sont frequemment mal encodes, ou objectif de recherche approximative (fuzzy matching).

**Choix retenu pour cet atelier :** on **conserve les accents**, le corpus etant globalement bien accentue et la distinction semantique etant importante pour la classification de sentiment.

### 3) et 4) Stop words a supprimer et a conserver

On part de la liste standard des stop words francais de NLTK (determinants, prepositions, pronoms, conjonctions : `"le"`, `"la"`, `"de"`, `"et"`, `"un"`, etc.), qui n'apportent pas de charge semantique utile a la classification de sentiment et peuvent etre **supprimes**.

En revanche, on **conserve les mots de negation** (`"ne"`, `"pas"`, `"non"`, `"jamais"`, `"aucun"`, `"rien"`, `"plus"`...) car ils **inversent le sens** d'une phrase : `"pas bon"` est l'oppose de `"bon"`. Les supprimer ferait perdre une information cruciale pour l'analyse de sentiment (un avis negatif pourrait etre mal classe comme positif).

In [ ]:
from nltk.corpus import stopwords

nltk.download("stopwords")

negations = {
    "ne", "pas", "non", "jamais", "aucun", "aucune", "rien",
    "personne", "plus", "ni", "sans", "guere"
}

stop_words_fr = set(stopwords.words("french")) - negations

print(f"Nombre de stop words NLTK (base) : {len(stopwords.words('french'))}")
print(f"Nombre de stop words retenus (negations exclues) : {len(stop_words_fr)}")
print(f"Negations conservees : {sorted(negations)}")

### 5) Stemming ou lemmatisation ?

Pour le francais, la **lemmatisation** est generalement preferable au stemming : elle ramene chaque mot a sa forme canonique reelle (infinitif pour un verbe, singulier masculin pour un adjectif) en s'appuyant sur la grammaire, alors que le stemming (ex: `SnowballStemmer('french')` de NLTK) tronque les mots de maniere heuristique et produit souvent des racines qui ne sont pas des mots reels, ce qui peut nuire a l'interpretabilite et a la qualite du vocabulaire.

NLTK ne propose pas de bon lemmatiseur francais natif. On utilise donc **spaCy** (modele `fr_core_news_sm`) si disponible, qui offre une lemmatisation francaise de qualite. En cas d'indisponibilite (modele non installe), on **replie sur le stemming NLTK** comme solution de secours, moins precise mais fonctionnelle.

In [ ]:
try:
    import spacy
    nlp_fr = spacy.load("fr_core_news_sm")
    methode = "lemmatisation (spaCy)"

    def normaliser_tokens(texte):
        doc = nlp_fr(texte)
        return [
            tok.lemma_ for tok in doc
            if tok.text not in stop_words_fr and tok.is_alpha
        ]
except (ImportError, OSError):
    from nltk.stem.snowball import SnowballStemmer
    stemmer = SnowballStemmer("french")
    methode = "stemming (NLTK SnowballStemmer, repli car spaCy fr indisponible)"

    def normaliser_tokens(texte):
        tokens = word_tokenize(texte, language="french")
        return [
            stemmer.stem(tok) for tok in tokens
            if tok not in stop_words_fr and tok.isalpha()
        ]

print(f"Methode retenue : {methode}")

exemple = df_clean["texte_norm"].iloc[0]
print(f"\nAvant : {exemple}")
print(f"Apres : {normaliser_tokens(exemple)}")

### 6) Creation de la colonne texte_final

In [ ]:
df_clean["texte_final"] = df_clean["texte_norm"].apply(
    lambda t: " ".join(normaliser_tokens(t))
)

df_clean[["texte_norm", "texte_final"]].head()

### 7) Verification texte, texte_clean et texte_final (10 premieres observations)

In [ ]:
df_clean[["texte", "texte_clean", "texte_final"]].head(10)

### 8) Sauvegarde du dataset nettoye (fin Partie 4)

In [ ]:
colonnes_finales = [
    "id_avis", "date", "source", "produit",
    "texte", "texte_clean", "texte_final",
    "sentiment", "note", "langue",
]

df_clean[colonnes_finales].to_csv(
    "../data/smart_reviews_cleaned.csv", index=False, encoding="utf-8"
)

print(f"Fichier sauvegarde : {len(df_clean)} lignes, {len(colonnes_finales)} colonnes")

## Partie 5 - Decoupage Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

df_model = pd.read_csv("../data/smart_reviews_cleaned.csv", encoding="utf-8")

train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42,
    stratify=df_model["sentiment"],
)

print(f"Taille train : {len(train_df)}")
print(f"Taille test  : {len(test_df)}")

print("\nProportions sentiment - donnees d'origine :")
print(df_model["sentiment"].value_counts(normalize=True).round(3))

print("\nProportions sentiment - train :")
print(train_df["sentiment"].value_counts(normalize=True).round(3))

print("\nProportions sentiment - test :")
print(test_df["sentiment"].value_counts(normalize=True).round(3))

## Partie 6 - Vectorisation

### 1) Bag of Words

Le vectoriseur est **fit uniquement sur `train_df`** puis applique (transform) a `train_df` et `test_df`, afin d'eviter toute fuite de donnees (le vocabulaire ne doit pas etre construit a partir d'informations issues du jeu de test).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(train_df["texte_final"].fillna(""))
X_test_bow = bow_vectorizer.transform(test_df["texte_final"].fillna(""))

vocab_bow = bow_vectorizer.get_feature_names_out()

print(f"Taille du vocabulaire (BoW) : {len(vocab_bow)}")
print(f"Extrait du vocabulaire : {list(vocab_bow[:20])}")
print(f"Shape matrice train : {X_train_bow.shape}")
print(f"Shape matrice test  : {X_test_bow.shape}")

pd.DataFrame(
    X_train_bow[:5, :10].toarray(),
    columns=vocab_bow[:10],
)

### 2) TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["texte_final"].fillna(""))
X_test_tfidf = tfidf_vectorizer.transform(test_df["texte_final"].fillna(""))

vocab_tfidf = tfidf_vectorizer.get_feature_names_out()

print(f"Taille du vocabulaire (TF-IDF) : {len(vocab_tfidf)}")
print(f"Extrait du vocabulaire : {list(vocab_tfidf[:20])}")
print(f"Shape matrice train : {X_train_tfidf.shape}")
print(f"Shape matrice test  : {X_test_tfidf.shape}")

pd.DataFrame(
    X_train_tfidf[:5, :10].toarray().round(3),
    columns=vocab_tfidf[:10],
)

### 3) TF-IDF avec unigrams et bigrams

In [ ]:
tfidf_ngram_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf_ngram = tfidf_ngram_vectorizer.fit_transform(train_df["texte_final"].fillna(""))
X_test_tfidf_ngram = tfidf_ngram_vectorizer.transform(test_df["texte_final"].fillna(""))

vocab_ngram = tfidf_ngram_vectorizer.get_feature_names_out()
bigrams_exemples = [mot for mot in vocab_ngram if " " in mot][:10]

print(f"Taille du vocabulaire (TF-IDF unigrams+bigrams) : {len(vocab_ngram)}")
print(f"Taille du vocabulaire (TF-IDF unigrams seuls)   : {len(vocab_tfidf)}")
print(f"Extrait du vocabulaire : {list(vocab_ngram[:15])}")
print(f"Exemples de bigrams : {bigrams_exemples}")
print(f"Shape matrice train : {X_train_tfidf_ngram.shape}")
print(f"Shape matrice test  : {X_test_tfidf_ngram.shape}")

pd.DataFrame(
    X_train_tfidf_ngram[:5, :10].toarray().round(3),
    columns=vocab_ngram[:10],
)

### 4) Comparaison des trois techniques de vectorisation

In [ ]:
comparaison = pd.DataFrame({
    "technique": ["Bag of Words", "TF-IDF (unigrams)", "TF-IDF (unigrams+bigrams)"],
    "taille_vocabulaire": [len(vocab_bow), len(vocab_tfidf), len(vocab_ngram)],
    "shape_train": [X_train_bow.shape, X_train_tfidf.shape, X_train_tfidf_ngram.shape],
    "shape_test": [X_test_bow.shape, X_test_tfidf.shape, X_test_tfidf_ngram.shape],
})
comparaison

**Comparaison :**

- **Bag of Words** capture les frequences brutes des mots, mais tend a sur-ponderer les mots tres frequents sans distinction de leur pouvoir discriminant (un mot present dans presque tous les avis, meme peu informatif, aura un poids eleve).
- **TF-IDF (unigrams)** pondere chaque mot selon son importance relative dans le corpus : il penalise les mots trop communs (peu informatifs) et met en avant les mots plus specifiques a un document. C'est generalement un choix plus performant que le BoW brut pour une tache de classification.
- **TF-IDF (unigrams+bigrams)** capture en plus du contexte local (ex: `"pas bon"` comme unite semantique distincte de `"pas"` et `"bon"` separement), ce qui est particulierement utile pour le sentiment ou la negation change le sens. En contrepartie, la taille du vocabulaire augmente fortement, ce qui accroit la sparsite de la matrice et le risque de surapprentissage sur un corpus de cette taille (~250 avis en train).

**Recommandation :** le **TF-IDF unigrams** constitue un bon compromis par defaut (performance/complexite). L'ajout des bigrams peut etre teste et conserve seulement s'il ameliore reellement les performances du modele sur le jeu de validation, en tenant compte du risque de sparsite sur ce corpus de taille modeste.